In [0]:
def write_user_feature_table(spark, db, table_names, run_date, window_size=365):
    sdf = spark.sql("""
                    INSERT OVERWRITE TABLE {db}.{mmc_user_features} PARTITION (partition_date = '{run_date}')
                    select user_id,
                    coalesce(last(platform), 'unknown') as platform,
                    coalesce(sum(store_visits) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as store_visits_30d,
                    coalesce(sum(store_visits) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as store_visits_90d,
                    coalesce(sum(pdp_views) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as pdp_views_30d,
                    coalesce(sum(pdp_views) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as pdp_views_90d,
                    coalesce(sum(tot_items_ordered) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as tot_items_ordered_30d,
                    coalesce(sum(tot_items_ordered) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as tot_items_ordered_90d,
                    coalesce(sum(tot_items_ordered) filter(where partition_date between date_sub('{run_date}', 180) and '{run_date}'), 0) as tot_items_ordered_180d,
                    coalesce(sum(app_items_ordered) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as app_items_ordered_30d,
                    coalesce(sum(app_items_ordered) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as app_items_ordered_90d,
                    coalesce(sum(app_items_ordered) filter(where partition_date between date_sub('{run_date}', 180) and '{run_date}'), 0) as app_items_ordered_180d,
                    coalesce(sum(coupons_owned) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as coupons_owned_30d,
                    coalesce(sum(coupons_owned) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as coupons_owned_90d,
                    coalesce(sum(coupons_owned) filter(where partition_date between date_sub('{run_date}', 180) and '{run_date}'), 0) as coupons_owned_180d,
                    coalesce(sum(coupons_redeemed) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as coupons_redeemed_30d,
                    coalesce(sum(coupons_redeemed) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as coupons_redeemed_90d,
                    coalesce(sum(coupons_redeemed) filter(where partition_date between date_sub('{run_date}', 180) and '{run_date}'), 0) as coupons_redeemed_180d
                    from {db}.{mmc_user_attributes}
                    where partition_date between date_sub('{run_date}', {window_size}) and '{run_date}' 
                    group by 1
                    """.format(db = db,
                               mmc_user_attributes = table_names['mmc_user_attributes'],
                               mmc_user_features = table_names['mmc_user_features'],
                               run_date = run_date,
                               window_size = window_size))
    return None

def write_product_feature_table(spark, db, table_names, run_date, window_size=365):
    sdf = spark.sql("""
                    INSERT OVERWRITE TABLE {db}.{mmc_product_features} PARTITION (partition_date = '{run_date}')
                    select product_id, 
                    product_type,
                    coalesce(sum(card_uv_imps) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as card_uv_imps_30d,
                    coalesce(sum(card_uv_imps) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as card_uv_imps_90d,
                    coalesce(sum(pdp_uv_imps) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as pdp_uv_imps_30d,
                    coalesce(sum(pdp_uv_imps) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as pdp_uv_imps_90d,
                    coalesce(sum(tot_order_cnt) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as tot_order_cnt_30d,
                    coalesce(sum(tot_order_cnt) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as tot_order_cnt_90d,
                    coalesce(sum(app_order_cnt) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as app_order_cnt_30d,
                    coalesce(sum(app_order_cnt) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as app_order_cnt_90d,
                    coalesce(avg(avg_order_msrp), 0) as avg_order_msrp,
                    coalesce(avg(avg_order_baseprice) filter(where partition_date between date_sub('{run_date}', 60) and '{run_date}'), 0) as avg_order_baseprice_60d,
                    coalesce(avg(avg_trans_price) filter(where partition_date between date_sub('{run_date}', 30) and '{run_date}'), 0) as avg_trans_price_30d,
                    coalesce(avg(avg_trans_price) filter(where partition_date between date_sub('{run_date}', 90) and '{run_date}'), 0) as avg_trans_price_90d
                    from {db}.{mmc_product_attributes}
                    where partition_date between date_sub('{run_date}', {window_size}) and '{run_date}' 
                    group by 1, 2
                    """.format(db = db,
                               mmc_product_attributes = table_names["mmc_product_attributes"],
                               mmc_product_features = table_names["mmc_product_features"],
                               run_date = run_date,
                               window_size = window_size))
    return None

In [0]:
import json
import pandas as pd

# Get pipeline parameters
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print("Input parameter: start_date {}".format(start_date))
print("Input parameter: end_date {}".format(end_date))

config_path = dbutils.widgets.get("table_config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

window_size = model_config['feature_agg_window_size']

In [0]:
date_range = [x.strftime('%Y-%m-%d') for x in pd.date_range(start_date, end_date)]
for run_date in date_range:
  print(f"Running for date: {run_date}")
  print("Writing user feature table...")
  write_user_feature_table(spark, db, table_names, run_date, window_size)
  print("Writing product feature table...")
  write_product_feature_table(spark, db, table_names, run_date, window_size)